# NLU HOSTAGE — ai_1_nlu_v1

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [2]:
import sys
print(sys.path[:3])  # lihat 3 entry pertama, cek apakah folder project ada di situ

import os
print(os.getcwd())  # cek working directory kernel sekarang

['/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload']
/Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy


In [12]:
import sys, os
print("CWD:", os.getcwd())
print("sys.path (3 pertama):", sys.path[:3])

CWD: /Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy
sys.path (3 pertama): ['/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.15_4/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload']


In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/janicetiffany/Downloads/prethesis-dev_andy")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import modules.nlu_training as nt
print(nt.__file__)  # pastikan sekarang nunjuk ke modules/nlu_training.py yang benar

/Users/janicetiffany/Downloads/prethesis-dev_andy/modules/nlu_training.py


In [9]:
from pathlib import Path
DATASET_FILENAME = "chat_dataset2_3intent.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v1"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")

# ==== Gabungkan label yang overlap ====
import pandas as pd

df = pd.read_csv(DATASET_PATH)   # ganti sesuai nama file kamu

MAPPING = {
    "offense": "offend",
    "defense": "defend",
    "neutral": "neutral"
}

df["label_intent"] = df["label_intent"].str.strip().str.lower().map(MAPPING)

unmapped = df.loc[df["label_intent"].isna(), "label_intent"].unique()
assert len(unmapped) == 0, f"Label belum di-mapping: {unmapped}"

df.to_csv("dataset_final3.csv", index=False, encoding="utf-8-sig")

print(df["label_intent"].value_counts())
print("Tersimpan: dataset_final3.csv |", len(df), "baris")


Dataset aktif: /Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy/data/chat_dataset2_3intent.csv
label_intent
offend     2561
neutral    2112
defend      942
Name: count, dtype: int64
Tersimpan: dataset_final3.csv | 5615 baris


In [10]:
from pathlib import Path
DATASET_FILENAME = "dataset_final3.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v1"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")

Dataset aktif: /Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy/data/dataset_final3.csv


In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cpu"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("offend",  "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),      # accusing
    ("defend",  "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),                       # defending
    ("offend",  "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),                          # bluffing
    ("neutral", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),                            # probing
    ("offend",  "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),          # deflecting
    ("defend",  "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),             # persuading
    ("defend",  "Klaim gw Civilian, gw gak punya skill malam."),                               # claiming
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),                      # neutral
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


CEK BERAPA VALUE PER INTENT

In [12]:
import pandas as pd
df = pd.read_csv(DATASET_PATH)
print(df['label_intent'].value_counts())

label_intent
offend     2561
neutral    2112
defend      942
Name: count, dtype: int64


PANJANGIN KOLOM

In [13]:
import pandas as pd
pd.set_option('display.max_colwidth', None)   # tampilkan isi kolom penuh, tanpa dipotong

In [10]:
import importlib
import modules.nlu_training
importlib.reload(modules.nlu_training)

<module 'modules.nlu_training' from '/Users/janicetiffany/Downloads/prethesis-dev_andy/modules/nlu_training.py'>

In [14]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
DATASET_PATH = DATASET_PATH
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=4476 | test=1120 | kelas=3

--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.8634
Macro F1    : 0.8380
Weighted F1 : 0.8623
              precision    recall  f1-score   support

      defend       0.79      0.71      0.75       188
     neutral       0.86      0.89      0.87       419
      offend       0.89      0.90      0.90       513

    accuracy                           0.86      1120
   macro avg       0.85      0.83      0.84      1120
weighted avg       0.86      0.86      0.86      1120

Model tersimpan: /Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy/models/intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline
Naive Bayes | train=4476 | test=1120 | kelas=3

--- Evaluasi Naive Bayes baseline (holdout test set) ---
Accuracy    : 0.7946
Macro F1    : 0.6746
Weighted F1 : 0.7605
              precision    recall  f1-score   support

      defend       0.90      0.20      0.32       188
     neutra

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 1120/1120 [00:00<00:00, 80571.15 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transformer indobenchmark/indobert-base-p1 | device=CPU | train=4476 | test=1120 | epoch=4


 25%|██▌       | 280/1120 [01:32<04:24,  3.18it/s]

{'loss': 0.5307, 'grad_norm': 3.3919475078582764, 'learning_rate': 1.665014866204163e-05, 'epoch': 1.0}


...

100%|██████████| 35/35 [00:03<00:00,  9.61it/s]



--- Evaluasi Transformer (holdout test set) ---
Accuracy    : 0.9116
Macro F1    : 0.8908
Weighted F1 : 0.9116
              precision    recall  f1-score   support

      defend       0.81      0.81      0.81       188
     neutral       0.92      0.91      0.92       419
      offend       0.94      0.95      0.94       513

    accuracy                           0.91      1120
   macro avg       0.89      0.89      0.89      1120
weighted avg       0.91      0.91      0.91      1120

Model Transformer tersimpan: /Users/janicetiffany/Downloads/prethesis-dev_andy/ai_1_nlu_v1 copy/models/intent_classifier_transformer
DEBUG jumlah artifacts: 3
DEBUG artifacts keys: ['SVM baseline', 'Naive Bayes baseline', 'IndoBERT Transformer']

--- 10 chat uji: SVM baseline ---
expected=offend       | predicted=offend       | confidence= 97.73% | OK
expected=defend       | predicted=defend       | confidence= 95.70% | OK
expected=defend       | predicted=defend       | confidence= 86.34% | OK
expecte

,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.9116,0.8908,0.9116,395.7,berhasil
1,SVM baseline,0.8634,0.8380,0.8623,0.2,berhasil
2,Naive Bayes baseline,0.7946,0.6746,0.7605,0.1,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,SVM baseline,9,0.9
1,IndoBERT Transformer,9,0.9
2,Naive Bayes baseline,7,0.7


In [39]:
import modules.nlu_training as nt
print(nt.__file__)

/Users/janicetiffany/Downloads/prethesis-dev_andy/modules/nlu_training.py


In [41]:
print(nt.HOSTAGE_BENCHMARK_TESTS)

(('accusing', 'B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman.'), ('defending', 'Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik.'), ('bluffing', 'Aku Spy dan semalam Guard Raka, jadi jangan curigai dia.'), ('probing', 'Stalker, kamu Peek siapa semalam dan apa hasil yang kamu lihat?'), ('deflecting', 'Jangan hanya fokus ke aku; cek D yang ceritanya berubah setiap ditanya.'), ('persuading', 'Vote C saja, dia paling diuntungkan ketika seseorang mendadak diam.'), ('claiming', 'Klaimku Civilian, jadi aku memang tidak memiliki aksi malam.'), ('neutral', 'Malam ini chat terkunci, kita lanjut diskusi setelah fase pagi.'), ('accusing', 'A selalu menghindar saat kita tanya kenapa dia menuduh tanpa bukti.'), ('defending', 'Diamku bukan pengakuan; aku masih bisa menjelaskan alibiku di fase siang.'))


In [42]:
with open("/Users/janicetiffany/Downloads/prethesis-dev_andy/modules/nlu_training.py") as f:
    content = f.read()
print("bluffing" in content)   # kalau True, berarti file di disk MASIH label lama
print("role_claim" in content) # kalau False, berarti edit belum ke-save

True
True


In [12]:
from modules.nlu_training import _split_dataset

x_train, x_test, y_train, y_test, report = _split_dataset(DATASET_PATH)

# Baru lihat contoh per label
for label in y_train.unique():
    print(f"\n=== {label} ===")
    print(x_train[y_train == label].head(5).tolist())


=== defend ===
['gw cuma ikut vote berdasarkan informasi yang ada bukan karena kerja sama hitman', 'fix bukan gw pelakunya semalem gw lagi jaga orang', 'fix yang nuduh gw itu hitman karena mau ngalihin perhatian', 'jangan jadikan gaya ngomongku sebagai bukti bahwa aku Hitman', 'gw claim Stalker dan hasil intipanku ada di tangan']

=== offend ===
['si H ngomongnya berubah setelah korban ditemukan berarti dia sedang menyesuaikan alibi', 'gw udah kasih semua info yang bisa gw kasih, sekarang sorot si Kevin yang selalu ikut mayoritas', 'si B tadi kena Gag Order pas mau jelasin sesuatu fix dia target Hitman', 'tuduhan lu ngawur banget si Fajar malah tiba-tiba ikut vote tanpa alasan', 'tuduhan ini tidak berdasar, apakah kamu sudah menjelaskan tindakanmu pada fase malam']

=== neutral ===
['siapa yang belum pernah ditanya soal aksi malamnya', 'lah langsung dieksekusi dong', 'Spy tadi malam nge-guard siapa? kasih clue dikit', 'Stalker sudah peek siapa aja sampai sekarang', 'kenapa lu yakin or

In [ ]:
try:
    import torch, transformers, datasets
    print("torch:", torch.__version__)
    print("transformers:", transformers.__version__)
    print("datasets:", datasets.__version__)
except ImportError as e:
    print("Masih belum lengkap:", e)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.14.0
transformers: 5.16.1
datasets: 5.0.1


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.6654 | 7/10 |
| Naive Bayes baseline | 0.6285 | 7/10 |
| IndoBERT Transformer (GPU) | 0.8464 | 3/10 |

Transformer terbaik pada holdout, tetapi SVM dan Naive Bayes lebih stabil pada 10 chat manual.